# Exercise 01 – Signals, noise, and channel capacity

An interactive model of a harmonic signal corrupted by Gaussian noise. The signal-to-noise ratio and the channel capacity are computed with functions that you implement yourself; the model then serves for experiments with real audio and image data. Theory and task descriptions are in the [README](../README.md).

## Imports

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import matplotlib.image as image
import numpy as np
import panel as pn
from bokeh.models import Button
from IPython.display import Audio, display
from scipy.io import wavfile

from lib.core import HarmSignal, NoiseSignal
from lib.params import add_logo, rc_params

pn.extension()
plt.rcParams.update(rc_params)

## Constants

In [ ]:
rng = np.random.default_rng()

NO_SAMPLES = 1000  # samples per signal over one period of 2π
TEXT_STYLE = {"font-size": "20px"}
IMAGE_PATH = "../fig/android_gray.jpeg"

## Theory

### A harmonic signal and Gaussian noise

`HarmSignal` generates $y(t) = A \sin(2 \pi f t + \varphi)$ on $t \in [0, 2\pi]$; `NoiseSignal` draws independent samples from the normal distribution with zero mean and standard deviation equal to its amplitude. Adding the two objects links them, so that the combined signal follows every change made with the sliders.

In [ ]:
signal = HarmSignal(f=np.sin, amplitude=1, freq=1, no_samples=NO_SAMPLES, title="Signal")
noise = NoiseSignal(f=rng.normal, amplitude=0.1, no_samples=NO_SAMPLES, title="Noise")
combined = signal + noise

## Exercise

### Task 1 – Implement the channel metrics

1. Implement `calc_signal_power(signal)` returning the mean power of a sampled signal.
2. Implement `calc_channel_capacity(S, N, B)` returning the Shannon–Hartley capacity for signal power `S`, noise power `N`, and bandwidth `B`.
3. Run the dashboard in Step 1 and verify the displayed values against a manual computation for one slider setting.

Both formulas are in the README. Keep the signatures below; the dashboard calls them by name.

In [ ]:
def calc_signal_power(signal: np.ndarray) -> float:
    """Mean power of a sampled signal [W]."""
    signal_power = np.nan
    return signal_power


def calc_channel_capacity(S: float, N: float, B: float) -> float:
    """Channel capacity [bit/s] for signal power S [W], noise power N [W], and bandwidth B [Hz]."""
    channel_capacity = np.nan
    return channel_capacity

### Step 1 – Model a signal with noise

Vary the parameters with the sliders and observe the combined signal. Before pressing anything, estimate qualitatively how the signal-to-noise ratio and the channel capacity should react to each slider. The *Calculate* button then reads the current signal and noise and displays the powers, the SNR, and the capacity computed by your functions; the bandwidth slider sets $B$.

In [ ]:
signal_power = pn.pane.Markdown(f"{np.nan}", styles=TEXT_STYLE)
noise_power = pn.pane.Markdown(f"{np.nan}", styles=TEXT_STYLE)
snr = pn.pane.Markdown(f"{np.nan}", styles=TEXT_STYLE)
channel_capacity = pn.pane.Markdown(f"{np.nan}", styles=TEXT_STYLE)


def calculate(event):
    S = calc_signal_power(signal.y)
    N = calc_signal_power(noise.y)
    signal_power.object = f"{S:.3f}"
    noise_power.object = f"{N:.3f}"
    snr.object = f"{S / N:.3f}"
    channel_capacity.object = f"{calc_channel_capacity(S, N, signal.band.value):.3f}"


button = Button(label="Calculate", button_type="primary")
button.on_click(calculate)

pn.Column(
    pn.Row(
        pn.Column(signal.amplitude, signal.freq, signal.phase, signal.band),
        signal.plot,
        combined.plot,
    ),
    pn.Row(
        pn.Column(noise.amplitude),
        noise.plot,
        pn.Column(
            pn.Spacer(height=20),
            button,
            pn.GridBox(
                pn.pane.Markdown("**Signal power:**", styles=TEXT_STYLE), signal_power, pn.pane.Markdown("W", styles=TEXT_STYLE),
                pn.pane.Markdown("**Noise power:**", styles=TEXT_STYLE), noise_power, pn.pane.Markdown("W", styles=TEXT_STYLE),
                pn.pane.Markdown("**SNR:**", styles=TEXT_STYLE), snr, pn.pane.Markdown("[–]", styles=TEXT_STYLE),
                pn.pane.Markdown("**Channel capacity:**", styles=TEXT_STYLE), channel_capacity, pn.pane.Markdown("bit/s", styles=TEXT_STYLE),
                ncols=3,
            ),
        ),
    ),
)

### Reference implementation

Run this cell only after Task 1 is complete. It replaces your functions with the reference versions from `lib/core.py`; press *Calculate* again and compare.

In [ ]:
from lib.core import calc_channel_capacity, calc_signal_power

### Task 2 – Apply noise to audio

1. Record or generate a short speech signal and save it as a mono 16-bit WAV file (`ffmpeg -i recording.m4a -ar 8000 -ac 1 -sample_fmt s16 input.wav` converts most formats).
2. Add Gaussian noise of increasing standard deviation and listen to each version.
3. Compute the SNR of each version with your `calc_signal_power` and relate it to what you hear.

The template loads the file, adds noise of one strength, plays the result, and prints the SNR; extend it to a series of noise levels.

In [ ]:
fs, audio = wavfile.read("input.wav")
audio = audio.astype(np.float64) / 32768  # int16 → [-1, 1)

sigma = 0.05
noisy = audio + rng.normal(0, sigma, audio.shape)

print(f"SNR = {calc_signal_power(audio) / calc_signal_power(noisy - audio):.1f}")
display(Audio(audio, rate=fs))
display(Audio(noisy, rate=fs))

### Task 3 – Apply noise to an image

1. Load the grayscale image and add Gaussian noise with variance 0.01, 0.05, and 0.1 (the image is scaled to $[0, 1]$).
2. Display the original and the three noisy versions side by side.
3. Compute the SNR of each version. The same image returns in Exercise 04, where the degradation is measured with PSNR and SSIM.

Values outside $[0, 1]$ must be clipped before display.

In [ ]:
img = image.imread(IMAGE_PATH).astype(np.float64) / 255

variances = [0.01, 0.05, 0.1]

fig, axes = plt.subplots(1, len(variances) + 1, figsize=(16, 4))
add_logo(fig)
axes[0].imshow(img, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original", fontsize=15)
for ax, var in zip(axes[1:], variances):
    noisy_img = np.clip(img + rng.normal(0, np.sqrt(var), img.shape), 0, 1)
    ax.imshow(noisy_img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Variance {var}", fontsize=15)
for ax in axes:
    ax.set_axis_off()
plt.show()

## Questions

1. What is the unit in which SNR is commonly expressed, and how does it relate to the linear ratio used in the Shannon–Hartley formula?
2. How does MIMO affect the channel capacity, given that the bandwidth per stream is unchanged?
3. For the telephone channel in the README table, what capacity does the formula give, and how does it compare with the 64 kbit/s PCM stream?
4. What happens to a 5 kHz tone sampled at $f_s = 8$ kHz?